# 02 · 看見時間穿越：洩漏 vs 正確的 point-in-time

> **這本為什麼是 notebook**：時間穿越（data leakage）是**對照式**的教材——
> 「錯誤做法讓指標虛高」這件事，你必須把兩張表、兩條 ROC 曲線並排看才會有感。
> 原本 `01_point_in_time_demo.py` 只能用 15 行 docstring 散文解釋它，那是圖該做的事。
>
> **用哪份資料**：`datasets/toy_sensors.csv`（5 台機器 × 150 小時的感測時序）。
> Feast 沙盒用的 diabetes 資料**每位病患只有一列**，沒有時間維度可以穿越，看不出差別；
> 感測器時序才有「同一台機器在不同時刻有多筆讀值」的結構。
>
> **本 notebook 補上的東西**：課程大綱 Module 3 Lab 第 2 步承諾「對照『錯誤做法』看指標虛高」，
> 但沙盒腳本沒有實作那個對照。這裡把它補齊。

In [ ]:
from pathlib import Path


def find_course_root() -> Path:
    """從當前目錄往上找到 mlops-course 根目錄（含 datasets/ 的那層）。

    notebook 沒有 __file__，而且你可能從任何位置啟動 Jupyter，
    所以用「往上層找標記檔」定位，不寫死相對路徑。
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "datasets" / "iris.csv").exists():
            return base
    raise FileNotFoundError("找不到 mlops-course/datasets/，請在 mlops-course/ 之內開啟本 notebook")


ROOT = find_course_root()
print("course root =", ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

SEED = 42
WINDOW = 6  # 滾動視窗長度（小時）

sensors = (
    pd.read_csv(ROOT / "datasets" / "toy_sensors.csv", parse_dates=["event_timestamp"])
    .sort_values(["machine_id", "event_timestamp"])
    .reset_index(drop=True)
)
SENSOR_COLS = ["temperature", "vibration", "current"]

print(f"{sensors.machine_id.nunique()} 台機器 × 每台 {len(sensors) // sensors.machine_id.nunique()} 小時")
print(f"時間範圍：{sensors.event_timestamp.min()} → {sensors.event_timestamp.max()}")
print(f"故障率：{sensors.failure.mean():.1%}")
sensors.head()

## 1. 任務設定：預測性維護

**目標**：在 T 時刻**之前**，判斷「機台在 T 會不會故障」。

這個「之前」是整件事的關鍵。上線後你在 T-1 就必須做出決策——
**T 當下那一筆讀值，你還沒拿到**。任何用到它的模型，離線分數再漂亮，上線都會崩。

## 2. 看見時間軸：你「當時真的有」的是哪些讀值？

挑 machine_01 的一段，把某個查詢時刻 T 標出來，看清楚兩種特徵各自吃了哪些資料。

In [ ]:
one = sensors[sensors.machine_id == "machine_01"].reset_index(drop=True)
seg = one.iloc[40:70].reset_index(drop=True)  # 取 30 小時的片段
t_idx = 20                                     # 在片段中挑一個查詢時刻 T
t_time = seg.loc[t_idx, "event_timestamp"]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(seg.event_timestamp, seg.temperature, marker="o", ms=4,
        color="#999999", label="temperature reading")

# 正確版看得到的：T 之前的 WINDOW 小時
past = seg.iloc[t_idx - WINDOW:t_idx]
ax.scatter(past.event_timestamp, past.temperature, s=90, color="#54A24B",
           zorder=3, label=f"AVAILABLE at T: past {WINDOW}h")

# 洩漏版偷用的：T 當下那一筆
ax.scatter([t_time], [seg.loc[t_idx, "temperature"]], s=170, marker="X",
           color="#E45756", zorder=4, label="LEAKED: reading AT T")

ax.axvline(t_time, color="#E45756", ls="--", lw=1)
ax.axvspan(seg.event_timestamp.iloc[t_idx], seg.event_timestamp.iloc[-1],
           color="#E45756", alpha=0.06)
ax.text(seg.event_timestamp.iloc[t_idx + 1], ax.get_ylim()[1] * 0.99,
        "future — not available at decision time", va="top", fontsize=9, color="#E45756")
ax.set_xlabel("time"); ax.set_ylabel("temperature")
ax.set_title("machine_01: what you actually have when predicting failure at T")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

print(f"查詢時刻 T = {t_time}")

綠點是**上線時你手上真的有的資料**。紅叉那一筆在 T 當下才量到——
離線做實驗時它就躺在你的 CSV 裡，很容易不小心用下去。

## 3. 兩種特徵建構

| 版本 | 怎麼算 | 對應的真實錯誤 |
| :--- | :--- | :--- |
| **洩漏版** | 直接用 T 當下的讀值 | 把特徵表照 `machine_id` 一路 join，抓到「最新一筆」——那筆往往來自未來 |
| **正確版** | 只用 T **之前** WINDOW 小時的滾動均值 | 嚴格以查詢時刻切齊（Feast `get_historical_features` 的行為） |

In [ ]:
grouped = sensors.groupby("machine_id")
feat = sensors.copy()

for col in SENSOR_COLS:
    # 洩漏版：T 當下的原始讀值（上線時拿不到）
    feat[f"{col}__leak"] = feat[col]
    # 正確版：先 shift(1) 把「當下」排除，再取過去 WINDOW 小時的均值
    feat[f"{col}__ok"] = grouped[col].transform(
        lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean()
    )

feat = feat.dropna().reset_index(drop=True)
LEAK_COLS = [f"{c}__leak" for c in SENSOR_COLS]
OK_COLS = [f"{c}__ok" for c in SENSOR_COLS]

print(f"可用樣本：{len(feat)} 列")
feat[["machine_id", "event_timestamp", "temperature",
      "temperature__leak", "temperature__ok", "failure"]].head(8)

看第一列就懂了：`temperature__leak` 完全等於 `temperature`（當下讀值），
而 `temperature__ok` 是**前幾小時的平均**，跟當下那筆無關。

## 4. 訓練兩個模型，比指標

In [ ]:
def fit_and_score(cols: list[str], label: str) -> dict:
    """用指定的特徵欄訓練同一種模型，回傳測試集指標。"""
    X, y = feat[cols], feat["failure"]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=SEED, stratify=y
    )
    model = LogisticRegression(max_iter=2000, random_state=SEED).fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        "版本": label,
        "ROC AUC": round(roc_auc_score(y_te, proba), 3),
        "Accuracy": round(accuracy_score(y_te, model.predict(X_te)), 3),
        "_y": y_te, "_p": proba,
    }


# 圖例要用英文（教學機器不一定有中文字型），所以另外帶一組英文短名
leak = fit_and_score(LEAK_COLS, "洩漏：用 T 當下的讀值")
leak["_en"] = "leaked (reading AT T)"
ok = fit_and_score(OK_COLS, f"正確：只用 T 之前 {WINDOW}h")
ok["_en"] = f"correct (past {WINDOW}h only)"

pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in (leak, ok)])

In [ ]:
gap = leak["ROC AUC"] - ok["ROC AUC"]
print(f"AUC 虛高 = +{gap:.3f}（相對高估 {gap / ok['ROC AUC']:.0%}）")

## 5. 把差距畫出來

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for r, color in ((leak, "#E45756"), (ok, "#54A24B")):
    fpr, tpr, _ = roc_curve(r["_y"], r["_p"])
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{r['_en']} — AUC={r['ROC AUC']}")
axes[0].plot([0, 1], [0, 1], ls=":", color="grey")
axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate")
axes[0].set_title("ROC: leakage inflates the curve")
axes[0].legend(fontsize=8, loc="lower right")

names = ["leaked", "correct"]
aucs = [leak["ROC AUC"], ok["ROC AUC"]]
bars = axes[1].bar(names, aucs, color=["#E45756", "#54A24B"], width=0.5)
axes[1].set_ylim(0.5, 1.0); axes[1].set_ylabel("ROC AUC")
axes[1].set_title("The gap is the illusion")
for b, v in zip(bars, aucs):
    axes[1].text(b.get_x() + b.get_width() / 2, v + 0.008, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

## 6. 這段 AUC 差距是哪裡來的？

洩漏版偷看的是**造成標籤的那一筆讀值**。上線時 T 還沒發生，你不可能有它——
那段分數是**幻覺**，會在上線第一天蒸發。

但請注意：正確版並不是 0.5（亂猜）。它仍有實質預測力，來自**機台的長期水準差異**
（有些機器就是跑得比較燙、比較容易故障）——那是真訊號，上線後也拿得到。

> **所以「離線指標變差」不是退步，是終於誠實。**
> 一個誠實的 0.79 遠比一個會在上線當天崩掉的 0.94 值錢。

驗證一下「機台水準差異」確實存在：

In [ ]:
by_machine = sensors.groupby("machine_id").agg(
    temperature=("temperature", "mean"),
    failure_rate=("failure", "mean"),
).round(3)
print(by_machine.to_string())
print(f"\n機台平均溫度 vs 故障率的相關性：{by_machine.corr().iloc[0, 1]:.2f}")

## 7. 靠紀律不可靠 → 讓工具給你保證

上面那個 `shift(1)` 只要有人手滑忘記寫，洩漏就悄悄回來了，而且**離線指標會變好看**——
沒有人會來跟你抱怨。這種錯誤不會自己浮現。

這就是 **Feature Store 存在的理由**：把「只取 `event_timestamp <= 查詢時刻`」
從「工程師要記得的紀律」變成**基礎設施層級的保證**。

```python
# Feast 的 get_historical_features 天生只回溯、不穿越
store.get_historical_features(
    entity_df=entity_df,      # 每列 = (哪個 entity, 在哪個時刻)
    features=[...],
).to_df()
```

**接下來換介質**：跑同資料夾的腳本，看這個保證怎麼落地成三個動詞
（`get_historical_features` / `materialize` / `get_online_features`）：

```bash
python 00_prepare_data.py
cd feature_repo && feast apply && cd ..
python 01_point_in_time_demo.py
```

> 那支是腳本而非 notebook，因為它產出的 registry 與 online store 是**檔案交付物**，
> 而且 `feature_repo/feature_definition.py` 必須是 Feast 能 import 的模組。